<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Classes and Iterators

Bundling data and behaviour into classes: class and instance variables, methods, visibility conventions, inheritance, and the predefined method names that let your own objects behave like built-in ones.

---

## Contents

1. [Classes](#classes)
    - [Class Variables](#class-variables)
    - [Instance variables](#instance-variables)
    - [Visibility of Class and Instance Variables](#visibility-of-class-and-instance-variables)
    - [Methods](#methods)
    - [Visibility of Methods](#visibility-of-methods)
    - [Inheritance](#inheritance)
    - [Predefined method names](#predefined-method-names)
    - [Changing classes / objects at runtime](#changing-classes-objects-at-runtime)
2. [Iterators](#iterators)

<a id="classes"></a>
## 1. Classes

Classes in Python are defined using the keyword `class`. This keyword is followed by the name of this class and a colon ":". In the code block after the colon, variables and methods of this class may be defined. The position of the variable defines whether this variable is a [Class Variable](#class-variables) or a [Instance variable](#instance-variables).

After defining a class, instances of this class may be created. These operate just like built-in data types with mutable properties (see also [Multiple names and mutable](01_Data_Types_and_Variables.ipynb#multiple-names-and-mutable)), since variables in Python always are references to objects (aka instances). Therefore technically, there is no difference between "primitive" data types and class objects when being passed as parameters.

In [ ]:
class Car:
    pass # the definition of a class must not be empty: the pass statement solves this problem

x = Car() # creates an object of type Car and stores a reference (instance) in the variable x
print(str(type(x)) + ": " + str(x))

In [ ]:
class Car:
    pass

x = Car # missing parentheses: we do not create an object from the class, but use the class directly
del x
#print(x) # results in an error, since the name / the variable no longer exists
          # possibly the object has already been deleted as well, since the only variable
          # that referenced the object no longer exists and the garbage collector
          # was therefore able to become active

<a id="class-variables"></a>
### 1.1 Class Variables

Variables, which are not within a method, but rather defined within the method are class variables. These class variables exist as just once per class. All instances of this class share this variable. If one instance changes the value of a class variable, all other instances of this class will see the changed value.

In [ ]:
class Car:
    wheels = 4
    doors = 5

x = Car
print(str(type(x)) + ": " + str(x))
print(x.wheels)
print(x.doors)

y = Car
y.wheels = 6 # changes the value of the class value: x.wheels now sees value 6, too!
y.doors = 3
print(x.wheels)
print(x.doors)

In [ ]:
class Car:
    wheels = 4
    doors = 5

x = Car

del x.wheels
#print(x.wheels) # Will lead to an runtime error, as x.wheels is not available anymore.

<a id="instance-variables"></a>
### 1.2 Instance variables

If variables are defined within a method (e.g. within the constructor), this will create an instance variable. These are created in every instance; changing its value in one instance will not change the value in another instance of this class.

If a class contains a class variable and an instance variable of the same name, the type of access defines which variable is referenced: if accessing via the class, the class variable is used, otherwise the instance variable:

In [ ]:
class Car:
    def __init__(self, wheels, doors): # self is not a keyword.
                                       # The first parameter is the reference to the instance (the object)
                                       # It may be named in any way, however self is a good convention...
        self.wheels = wheels
        self.doors = doors

#x = Car() # Error, since arguments are missing
x = Car(4,5)
print(x.wheels)
print(x.doors)

In [ ]:
class Car:
    def __init__(self, wheels = 4, doors = 5): # with default values
        self.wheels = wheels
        self.doors = doors

x = Car()
y = Car(6,3)
print(x.wheels) # was not overwritten with 6: it's an instance variable
print(x.doors)
print(y.wheels)
print(y.doors)

In [ ]:
class Car:
    wheels = 6
    def __init__(self, wheels = 4):
        self.wheels = wheels # does not access the class variable, but rather creates a new instance variable!
        #Car.wheels = wheels # via the class name, the class variable may be accessed!

x = Car()
print(x.wheels)   # accessing the instance variable
print(Car.wheels) # accessing the class variable

In [ ]:
class Car:
    def __init__(self, wheels = 4):
        self.wheels = wheels
    
    def get_wheels(self):
        #return wheels # one cannot access the class/instance variable only by name 
                       # one must always use the class name or the passed name to the object
        return self.wheels

x = Car()
print(x.get_wheels())

<a id="visibility-of-class-and-instance-variables"></a>
### 1.3 Visibility of Class and Instance Variables

Python does not differentiate between private, public and protected like C++ or Java. All variables are public by default (and therefore mutable from outside of the class). However, there are conventions that signal to users of the class, whether a variable must only be used from within the class.

The same is true for [Visibility of methods](#visibility-of-methods).

In [ ]:
class Car:
    _wheels = 4 # One underscore: Variable should not be changed by users of this class (although it's technically possible)
    __doors = 5 # double underscore: Python changes the name of the variable: direct access therefore is not possible!
                # (The variable is prepended with a clean class name and a leading underscore, which will NOT make the
                # variable private. However, it will resolve naming conflicts.)
    __test1_ = 1 # One additional underscore as suffix: will change the name of the variable
    __test2__ = 2 # More than one additional underscore as suffix will prohibit changing the name

x = Car # missing parentheses: we do not create an instance, but rather reference the class itself
print(str(type(x)) + ": " + str(x))

print(x._wheels)
x._wheels = 6
print(x._wheels)

#print(x.__doors) # leads to an error, since __doors is not available (it's been renamed)
print(x.__dict__) # Shows the complete content of an instance / of the class
print(x._Car__doors) # Using this name, we may access the value of the "hidden" variable

# Once again, however, completely automated
d = x.__dict__
for key in d:
    if key.endswith("__doors"):
        print(d[key])

<a id="methods"></a>
### 1.4 Methods

Functions that are defined within a class are called its methods. Methods may either only be called through a object or through the object and class (classmethod/staticmethod). Their first parameter is either a reference to the object, the class (classmethod), or neither (staticmethod). The reference to the object or class is passed automatically when the method is called; it does not have to be listed manually in the parameter list.

Python Methods are virtual by default, i.e. methods of the same name of a base class are always overloaded (see [Inheritance](#inheritance)). Therefore a method of a base class that calls another method defined in the same base class may end up calling a method of a derived class that overloads it (if called from a derived class).

In [ ]:
class Car:
    def __init__(self, wheels = 4, doors = 5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self): # a simple Method: first parameter is always a reference to the object, from where the method was called
        print("A car with " + str(self.wheels) + " wheels " + str(self.doors) + " doors.")
    
    @classmethod
    def print_class(cls): # a class method: first parameter is always the class
        print("Class: " + str(cls))
    
    @staticmethod
    def print_static(): # a static method: neither object nor class is passed in
        print("Static text")

x = Car()
x.print()
x.print_class()
Car.print_class()
x.print_static()
Car.print_static()

x = Car(6,3)
x.print()
x.print_class()
Car.print_class()
x.print_static()
Car.print_static()

#Car.print() # returns a runtime error: calling the method missing the required positional argument self 
Car.print(x) # Now, the function has the reference to a proper object
             # => This is identical to calling x.print()!

In [ ]:
class Car:
    def __init__(self, wheels = 4, doors = 5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self, number): # After the reference to self, further parameters may follow
        print(str(number) + ": A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")
    
    @classmethod
    def print_class(cls, number): # Also after the reference to the class, further parameters may follow
        print(str(number) + ": Class: " + str(cls))
    
    @staticmethod
    def print_static(number):
        print(str(number) + ": Static text")

x = Car(6,3)
x.print(1)
x.print_class(2)
Car.print_class(3)
x.print_static(4)
Car.print_static(5)

In [ ]:
def add_wheel(self):
    self.wheels += 1

class Car:
    def __init__(self, wheels = 4, doors = 5):
        self.wheels = wheels
        self.doors = doors
    
    add_wheel = add_wheel # A method does not have to be defined within the class.
                          # An attribute with a reference suffices.
                          # However, this is frowned upon as it's confusing.
    
    def add_axle(self):
        self.add_wheel() # We may call methods from other methods
        self.add_wheel() # the calls is by reference.

x = Car()
print(x.wheels)
x.add_wheel()
print(x.wheels)
x.add_axle()
print(x.wheels)

In [ ]:
class Test:
    def test_method(self):
        print("test")

x = Test()
test_method = x.test_method

test_method()
print(test_method.__self__) # Using the reference to an method, we may access the method:
test_method.__self__.test_method()
print(test_method.__func__) # Using the reference to a method, we may access the function within a class
test_method.__func__(x)

<a id="visibility-of-methods"></a>
### 1.5 Visibility of Methods

As with variables, the same applies to methods: they are generally visible from outside a class or object. This can only be restricted by conventions.

See [Visibility of Class- and Instance-variables](#visibility-of-class-and-instance-variables).

In [ ]:
class Test:
    def __init__(self, i):
        #self.add(i)
        self.__add(i)
    
    def add(self, i):
        self.i = i
    
    def __str__(self):
        return str(self.i)
    
    __add = add # private copy with a new name may use __init__, this will show up in the dict below as _Test__add!

class Sub_Test(Test):
    def add(self, number, i): # Overload the add method with an additional parameter, but not the __add() method!
        self.i = str(number) + ": " + str(i)

x = Sub_Test(2)
print(Test.__dict__)
print(Sub_Test.__dict__) # Please not the difference in the methods, __add is still visible via x._Test__add()
print(x.__dict__)
print(x)
x.add(2, 42)
print(x)

<a id="inheritance"></a>
### 1.6 Inheritance

Python supports single and multiple base class inheritance. Since instance variables may be defined inside of methods within a class, overloading of methods may remove instance variables (which other methods might erroneously try to access, leading to runtime errors). The class name of the base class or super functions may be used to call the method of the base class from a overloaded method.

When using multiple base classes the name of a variable or a method may already be defined in more than one base class. Resolving the name is done in the order of the base-classes when the derived class was defined. The first class containing the name in question will be referenced. All base classes are subject to depth-first, left-to-right search (not searching twice in the same class where there is an overlap in the inheritance hierarchy).

See [Python 3 docs on Multiple Inheritance.](https://docs.python.org/3/tutorial/classes.html#multiple-inheritance)

The built-in data types may be used as base classes, themselves.

In [ ]:
class Car:
    def __init__(self, wheels=4, doors=5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self):
        print("A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")

class Bus(Car): # Bus is derived from car an inherits all methods and variables.
    seats = 20

x = Bus()
print(str(type(x)) + ": " + str(x))

x.print()
print(x.seats)

In [ ]:
class Car:
    def __init__(self, wheels=4, doors=5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self):
        print("A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")

class Bus(Car):
    def __init__(self, seats=20): # overwrites the init-method of the base class (Car)
        self.seats = seats

x = Bus()

x.print() # leads to an error, since wheels and doors only defined in the init-method of the base class

In [ ]:
class Car:
    def __init__(self, wheels=4, doors=5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self):
        print("A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")

class Bus(Car):
    def __init__(self, wheels=4, doors=5, seats=20):
        self.seats = seats
        #Car.__init__(self, wheels, doors) # calls the init method of the base class
        super().__init__(wheels, doors) # Better: calls the init method of the base class without having to repeat the base-class name.

x = Bus()

x.print() # Now, all the required variables are initialized (but since Car doesn't know anything about seats, this information is lost)

In [ ]:
class Car:
    def __init__(self, wheels=4, doors=5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self):
        print("A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")

class Bus(Car):
    def __init__(self, wheels=4, doors=5, seats=20):
        self.seats = seats
        super().__init__(wheels, doors)

class Test1:
    def print(self):
        print("This is a Test")

class Test2:
    pass

for o in Car(4,5), Bus(6,3), Test1():
    o.print() # Duck Typing: print may be called for Car, Bus and Test1, but not Test2
              # Even though Car, Bus and Test1 do not share the same base class

In [ ]:
class Car:
    def __init__(self, wheels=4, doors=5):
        self.wheels = wheels
        self.doors = doors
    
    def print(self):
        print("A car with " + str(self.wheels) + " wheels and " + str(self.doors) + " doors.")

class Bus(Car):
    def __init__(self, wheels=6, doors=3, seats=20):
        self.seats = seats
        super().__init__(wheels, doors)

class Test1:
    def print(self):
        print("This is a test")

class Multi(Bus, Test1): # The order does matter (see below)
    pass

x = Multi()

x.print() # Using the rules above, multiple inheritance is deterministic, even though there are multiple instance of print:
          # the order in which Multi derived from its base classes determines, which print function is called.
          # Here Multi derives from Bus, which  inherits print() from Car,
          # and since Bus is inherited earlier than Test1 -- however the initialization of values happens in Bus' init().

In [ ]:
class Number(int):
    def __str__(self):
        return "Number " + super().__str__()

x = Number()
print(Number.__dict__)
print(x.__dict__)
print(x)

x = Number(2)
print(x)

<a id="predefined-method-names"></a>
### 1.7 Predefined method names

Python provides several predefined method names. Implementing methods carrying this name influences the behaviour of instances created out of such classes. For example using \_\_init\_\_ one may define a constructor, while \_\_eq\_\_ defines the behaviour of the == operator.

The following examples present a few of the predefined methods:

Further method names, which directly influence the behaviour are listed in https://docs.python.org/3/reference/datamodel.html#basic-customization

In [ ]:
class Test:
    def __new__(cls): # creates a new object, but does not initialize it
        print("in new")
        return super().__new__(cls) # has to return the new object (only then __init__ will be called next)
    
    def __init__(self): # initializes a new object (and may only return None)
        print("in init")
    
    def __del__(self): # Once the Garbage Collector frees this object it calls __del__
        print("in del")
    
    def __str__(self): # This method is called by str(), format() and print() to return a descriptive text of this object
        return "Just a test"

x = Test()

print(x)

del x

In [ ]:
class Bus():
    def __init__(self, seats):
        self.seats = seats
    
    def __eq__(self, other): # Called as == operator
        if self.seats == other.seats:
            return True
        else:
            return False
    
    def __lt__(self, other): # called as < operator
        if self.seats < other.seats:
            return True
        else:
            return False

a = Bus(10)
b = Bus(20)

if a == b:
    print("Bus a and b have the same number of seats")
elif a < b:
    print("Bus a has less seats than b")
else:
    print("Bus a hat more seats than b")

<a id="changing-classes-objects-at-runtime"></a>
### 1.8 Changing classes / objects at runtime

In Python classes and object may be changed at runtime. Any attributes (Classes, instance variables, even methods) may be changed, added or removed at runtime.

In [ ]:
class Test:
    a = 3
    
    def __init__(self, b = 3):
        self.b = b
    
    def get_value(self):
        return self.a * self.b

x = Test()
print(Test.__dict__)
print(x.__dict__)
print("x.a * x.b = " + str(x.get_value()))

print()
print("Going to remove classvariable a and instance variable b:")
del Test.a # In order to use del, the name of the variable to be deleted has to be known at the time of writing this source
del x.b
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to add a class variable a and an instance variable b:")
Test.a = 5
x.b = 5
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to remove class variable a and an instance variable b using delattr:")
delattr(Test, "a") # delattr's first parameter is a string, i.e. the name of the variable may be generated at run-time
delattr(x, "b")
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to add a class variable a and an instance variable b using setattr:")
setattr(Test, "a", 7)
setattr(x, "b", 7)
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to remove a method:")
#del x.get_value # Methods of a Class may only be removed from the class
                 # (which changes all objects instantiated from this class),
                 # but cannot be removed from single objects.
del Test.get_value
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to add methods:")
Test.get_value = lambda self : self.a + self.b # instead of a lambda and already defined function would work, too
x.get_value_from_instance = (lambda self : self.a / self.b).__get__(x)  # In order to have "self" being passed to the method
                                                                        # this method has to be bound to the object
                                                                        # (which is accomplished using __get__)
print(Test.__dict__)
print(x.__dict__)

print("x.a + x.b = " + str(x.get_value()))
print("x.a / x.b = " + str(x.get_value_from_instance()))

print()
print("Going to remove methods using delattr:")
delattr(x, "get_value_from_instance") # Methods bound to an object
                                      # may be removed from this object
delattr(Test, "get_value")
print(Test.__dict__)
print(x.__dict__)

print()
print("Going to add method using setattr:")
def add_in_class_method(self):
    return "in Class: " + str(self.a + self.b)
def add_in_instance_method(self):
    return "in instance: " + str(self.a + self.b)
setattr(Test, "add_in_class_method", add_in_class_method)
setattr(x, "add_in_instance_method", add_in_instance_method.__get__(x))
print(Test.__dict__)
print(x.__dict__)

print(x.add_in_class_method())
print(x.add_in_instance_method())

<a id="iterators"></a>
## 2. Iterators

For loops iterate over sequences of elements or over generators. This is implemented by the \_\_iter\_\_ method inherent to any sequence and generator.
This method returns an object, which itself provides the \_\_next\_\_ method, which may be called using `next()`, providing the next element in the sequence or generator. If there is no further element, the exception `StopIteration` is thrown and the loop terminates.

The methods \_\_iter\_\_ and \_\_next\_\_ may be used to employ ones classes in for loops.

In [ ]:
class Every_nth_element:
    def __init__(self, n, list):
        self._n = n
        self._list = list
        self._pos = -1
        self._len = len(list)
    
    def __iter__(self):
        return self
    
    def __next__(self):
        self._pos += self._n
        if (self._pos >= self._len):
            raise StopIteration
        return self._list[self._pos]

x = (1, 2, 3, 4, 5, 6, 7)
for i in Every_nth_element(2, x):
    print(i)

In [ ]:
def every_nth_element(n, list): # often generators may be used instead of iterator
    for i in list[1::n]:
        yield i # every call to next executes the yield until the StopIteration Exception is thrown

x = (1, 2, 3, 4, 5, 6, 7)
for i in every_nth_element(2, x): # Generator automatically provides __iter__ and __next__
    print(i)

In [ ]:
x = (1, 2, 3, 4, 5, 6, 7)
for i in x[1::2]: # of course, in this particular instance the above slice operation may be used directly
    print(i)